# Pipes - Despliegue

- Cargamos el modelo
- Cargamos los datos futuros
- Aplicar tubería

In [1]:
#Cargamos librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Función para eliminar atípicos
def limpiar_atipicos(X):
    """
    Limpia valores atípicos reemplazándolos con np.nan
    """
    # Limpiar LotFrontage (valores >= 270)
    X.loc[X["LotFrontage"] >= 270, "LotFrontage"] = np.nan

    # Limpiar TotalBsmtSF (valores > 5500)
    X.loc[X["TotalBsmtSF"] > 5500, "TotalBsmtSF"] = np.nan

    # Limpiar BsmtFinSF1 (valores > 5500)
    X.loc[X["BsmtFinSF1"] > 5500, "BsmtFinSF1"] = np.nan

    return X

print('Función de limpieza de atípicos definida')


Función de limpieza de atípicos definida


In [3]:
import pickle
with open('pipeline_final.pkl', 'rb') as f:
    pipeline_final = pickle.load(f)

pipeline_final

Pipeline(steps=[('preprocesamiento',
                 Pipeline(steps=[('limpiar_atipicos',
                                  FunctionTransformer(func=<function limpiar_atipicos at 0x7892e6c74d60>)),
                                 ('preprocessor',
                                  ColumnTransformer(transformers=[('num',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer())]),
                                                                   ['LotFrontage',
                                                                    'TotalBsmtSF',
                                                                    'BsmtFinSF1',
                                                                    'MasVnrArea']),
                                                                  ('cat',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(st...
                                                                    'Condition2',
                                                                    'BldgType',
                                                                    'HouseStyle',
                                                                    'RoofStyle',
                                                                    'RoofMatl',
                                                                    'Exterior1st',
                                                                    'Exterior2nd',
                                                                    'MasVnrType',
                                                                    'ExterQual',
                                                                    'ExterCond',
                                                                    'Foundation',
                                                                    'BsmtQual',
                                                                    'BsmtCond',
                                                                    'BsmtExposure',
                                                                    'BsmtFinType1',
                                                                    'BsmtFinType2',
                                                                    'Heating',
                                                                    'HeatingQC',
                                                                    'CentralAir',
                                                                    'Electrical', ...])]))])),
                ('modelo',
                 RandomForestRegressor(min_samples_leaf=4, n_estimators=200,
                                       random_state=42))])

In [4]:
#Cargamos los datos futuros
data = pd.read_csv("/content/datos_futuro_vivienda.csv")
data.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,YearBuilt_Category,YearRemodAdd_Category
0,60,RL,65,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,NaN,NaN,NaN,0,2,2008,WD,Normal,1981-2010,1981-2010
1,20,RL,80,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,NaN,NaN,NaN,0,5,2007,WD,Normal,1951-1980,1950-1980
2,60,RL,68,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,NaN,NaN,NaN,0,9,2008,WD,Normal,1981-2010,1981-2010


In [7]:
import streamlit as st
import pandas as pd

st.set_page_config(page_title="Predicción de precio de una casa", layout="wide")
st.title('Predicción de precio de una casa')

# Opciones para los campos importantes:
MSZoning_opts = ['RL', 'RM', 'C (all)', 'FV', 'RH']
Neighborhood_opts = [
    'CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst', 'NWAmes', 'OldTown',
    'BrkSide', 'Sawyer', 'NridgHt', 'NAmes', 'SawyerW', 'IDOTRR', 'MeadowV', 'Edwards',
    'Timber', 'Gilbert', 'StoneBr', 'ClearCr', 'NPkVill', 'Blmngtn', 'BrDale', 'SWISU', 'Northridge'
]

# SOLO ESTOS 7 campos visibles para el usuario:
entrada = {
    'MSZoning': st.selectbox('Clasificación de zona (MSZoning)', MSZoning_opts, index=0),
    'Neighborhood': st.selectbox('Barrio (Neighborhood)', Neighborhood_opts, index=0),
    'GrLivArea': st.number_input('Área habitable en pies cuadrados (GrLivArea)', min_value=200, max_value=7000, value=1500),
    'LotArea': st.number_input('Área del lote (LotArea)', min_value=1000, max_value=100000, value=8450),
    'OverallQual': st.slider('Calidad general (OverallQual)', min_value=1, max_value=10, value=6),
    'FullBath': st.slider('Baños completos (FullBath)', min_value=0, max_value=3, value=1),
    'BedroomAbvGr': st.slider('Habitaciones (BedroomAbvGr)', min_value=0, max_value=8, value=3),
}

# RELLENA EL RESTO de variables del pipeline con defaults (usualmente, los valores más frecuentes del dataset de entrenamiento)
entrada_defaults = {
    'MSSubClass': 20,
    'LotFrontage': 60,
    'Street': 'Pave',
    'Alley': 'NA',
    'LotShape': 'Reg',
    'LandContour': 'Lvl',
    'Utilities': 'AllPub',
    'LotConfig': 'Inside',
    'LandSlope': 'Gtl',
    'Condition1': 'Norm',
    'Condition2': 'Norm',
    'BldgType': '1Fam',
    'HouseStyle': '1Story',
    'OverallCond': 5,
    'RoofStyle': 'Gable',
    'RoofMatl': 'CompShg',
    'Exterior1st': 'VinylSd',
    'Exterior2nd': 'VinylSd',
    'MasVnrType': 'None',
    'MasVnrArea': 0,
    'ExterQual': 'TA',
    'ExterCond': 'TA',
    'Foundation': 'PConc',
    'BsmtQual': 'TA',
    'BsmtCond': 'TA',
    'BsmtExposure': 'No',
    'BsmtFinType1': 'Unf',
    'BsmtFinSF1': 0,
    'BsmtFinType2': 'Unf',
    'BsmtFinSF2': 0,
    'BsmtUnfSF': 0,
    'TotalBsmtSF': 0,
    'Heating': 'GasA',
    'HeatingQC': 'Ex',
    'CentralAir': 'Y',
    'Electrical': 'SBrkr',
    '1stFlrSF': 0,
    '2ndFlrSF': 0,
    'LowQualFinSF': 0,
    'BsmtFullBath': 0,
    'BsmtHalfBath': 0,
    'HalfBath': 0,
    'KitchenAbvGr': 1,
    'KitchenQual': 'TA',
    'TotRmsAbvGrd': 6,
    'Functional': 'Typ',
    'Fireplaces': 0,
    'FireplaceQu': 'NA',
    'GarageType': 'Attchd',
    'GarageYrBlt': 2000,
    'GarageFinish': 'Unf',
    'GarageCars': 1,
    'GarageArea': 200,
    'GarageQual': 'TA',
    'GarageCond': 'TA',
    'PavedDrive': 'Y',
    'WoodDeckSF': 0,
    'OpenPorchSF': 0,
    'EnclosedPorch': 0,
    '3SsnPorch': 0,
    'ScreenPorch': 0,
    'PoolArea': 0,
    'PoolQC': 'NA',
    'Fence': 'NA',
    'MiscFeature': 'NA',
    'MiscVal': 0,
    'MoSold': 6,
    'YrSold': 2010,
    'SaleType': 'WD',
    'SaleCondition': 'Normal',
    'YearBuilt_Category': '1981-2010',
    'YearRemodAdd_Category': '1981-2010',
}

# Combina los campos pedidos con los defaults (lo del usuario prevalece)
for k,v in entrada_defaults.items():
    if k not in entrada:
        entrada[k] = v

data = pd.DataFrame([entrada])

if st.button('Predecir precio'):
    pred = pipeline_final.predict(data)
    st.success(f"El precio estimado de la casa es: ${pred[0]:,.2f}")
    st.write("Datos ingresados:")
    st.dataframe(data)

2025-11-12 01:27:46.366 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-12 01:27:46.367 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-12 01:27:46.525 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-11-12 01:27:46.526 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-12 01:27:46.527 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-12 01:27:46.529 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-12 01:27:46.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

La prediccion tiene un error con respecto al precio de aprox 19400 dolares

In [5]:
#Hacemos la predicción con el Tree
Y_rf = pipeline_final.predict(data)
print(Y_rf)

[208723.64086236 175665.26651338 259644.5096665 ]


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [6]:
data['Prediccion']=Y_rf
data

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,YearBuilt_Category,YearRemodAdd_Category,Prediccion
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,NaN,NaN,0,2,2008,WD,Normal,1981-2010,1981-2010,208723.640862
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,NaN,NaN,0,5,2007,WD,Normal,1951-1980,1950-1980,175665.266513
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,NaN,NaN,0,9,2008,WD,Normal,1981-2010,1981-2010,259644.509667
